# SpaceX Falcon 9 – Exploratory Data Analysis with Visualization

This notebook creates the six EDA visualizations required for the final Capstone presentation.

**Input:** `dataset_part_2.csv`  
**Required columns:** `FlightNumber`, `PayloadMass`, `Orbit`, `LaunchSite`, `Class`  

Run the cells from top to bottom. Each figure is also saved as a PNG file for direct insertion into PowerPoint.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 120

DATA_FILE = "dataset_part_2.csv"
if not os.path.exists(DATA_FILE):
    raise FileNotFoundError(
        f"{DATA_FILE} was not found. Upload it to the Colab/Jupyter working directory first."
    )

df = pd.read_csv(DATA_FILE)
print("Dataset shape:", df.shape)
display(df.head())


## 1. Validate and prepare the dataset


In [ ]:
required = ["FlightNumber", "PayloadMass", "Orbit", "LaunchSite", "Class"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}. Available columns: {list(df.columns)}")

df["FlightNumber"] = pd.to_numeric(df["FlightNumber"], errors="coerce")
df["PayloadMass"] = pd.to_numeric(df["PayloadMass"], errors="coerce")
df["Class"] = pd.to_numeric(df["Class"], errors="coerce")

df_eda = df.dropna(subset=["FlightNumber", "PayloadMass", "Orbit", "LaunchSite", "Class"]).copy()
df_eda["Class"] = df_eda["Class"].astype(int)

print("Rows used for EDA:", len(df_eda))
print("Success rate:", round(df_eda["Class"].mean() * 100, 1), "%")
display(df_eda[required].head())


## 2. Flight Number vs. Launch Site

This plot examines whether later flights show a higher landing success rate at each launch site.


In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
sns.scatterplot(data=df_eda, x="FlightNumber", y="LaunchSite", hue="Class", palette={0:"#d62728", 1:"#2ca02c"}, s=85, ax=ax)
ax.set_title("Flight Number vs. Launch Site")
ax.set_xlabel("Flight Number")
ax.set_ylabel("Launch Site")
ax.legend(title="Landing Class", labels=["Failure (0)", "Success (1)"])
plt.tight_layout()
plt.savefig("EDA_01_FlightNumber_vs_LaunchSite.png", bbox_inches="tight")
plt.show()


## 3. Payload Mass vs. Launch Site

This plot compares payload mass across launch sites and distinguishes successful from unsuccessful first-stage landings.


In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
sns.scatterplot(data=df_eda, x="PayloadMass", y="LaunchSite", hue="Class", palette={0:"#d62728", 1:"#2ca02c"}, s=85, ax=ax)
ax.set_title("Payload Mass vs. Launch Site")
ax.set_xlabel("Payload Mass (kg)")
ax.set_ylabel("Launch Site")
ax.legend(title="Landing Class", labels=["Failure (0)", "Success (1)"])
plt.tight_layout()
plt.savefig("EDA_02_Payload_vs_LaunchSite.png", bbox_inches="tight")
plt.show()


## 4. Success Rate vs. Orbit Type

The success rate is calculated as the mean of the binary `Class` variable for each orbit.


In [ ]:
orbit_success = (df_eda.groupby("Orbit", as_index=False)["Class"].mean()
                 .rename(columns={"Class":"SuccessRate"})
                 .sort_values("SuccessRate", ascending=False))

fig, ax = plt.subplots(figsize=(11, 6))
sns.barplot(data=orbit_success, x="Orbit", y="SuccessRate", color="#4472C4", ax=ax)
ax.set_title("Landing Success Rate by Orbit Type")
ax.set_xlabel("Orbit")
ax.set_ylabel("Success Rate")
ax.set_ylim(0, 1.05)
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig("EDA_03_SuccessRate_vs_Orbit.png", bbox_inches="tight")
plt.show()
display(orbit_success)


## 5. Flight Number vs. Orbit Type

This plot examines whether accumulated flight experience is associated with landing success for different orbit types.


In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))
sns.scatterplot(data=df_eda, x="FlightNumber", y="Orbit", hue="Class", palette={0:"#d62728", 1:"#2ca02c"}, s=80, ax=ax)
ax.set_title("Flight Number vs. Orbit Type")
ax.set_xlabel("Flight Number")
ax.set_ylabel("Orbit")
ax.legend(title="Landing Class", labels=["Failure (0)", "Success (1)"])
plt.tight_layout()
plt.savefig("EDA_04_FlightNumber_vs_Orbit.png", bbox_inches="tight")
plt.show()


## 6. Payload Mass vs. Orbit Type

This plot shows how payload mass and landing outcome vary among orbit types.


In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))
sns.scatterplot(data=df_eda, x="PayloadMass", y="Orbit", hue="Class", palette={0:"#d62728", 1:"#2ca02c"}, s=80, ax=ax)
ax.set_title("Payload Mass vs. Orbit Type")
ax.set_xlabel("Payload Mass (kg)")
ax.set_ylabel("Orbit")
ax.legend(title="Landing Class", labels=["Failure (0)", "Success (1)"])
plt.tight_layout()
plt.savefig("EDA_05_Payload_vs_Orbit.png", bbox_inches="tight")
plt.show()


## 7. Launch Success Yearly Trend

If the dataset has a usable date or year column, the code calculates average landing success by year. The code supports common column names used in the Capstone labs.


In [ ]:
year = None
if "Date" in df.columns:
    year = pd.to_datetime(df["Date"], errors="coerce").dt.year
elif "date" in df.columns:
    year = pd.to_datetime(df["date"], errors="coerce").dt.year
elif "Year" in df.columns:
    year = pd.to_numeric(df["Year"], errors="coerce")
elif "year" in df.columns:
    year = pd.to_numeric(df["year"], errors="coerce")

if year is None or year.notna().sum() == 0:
    raise ValueError("No usable Date/Year column found. Add the launch date column to dataset_part_2.csv for the yearly trend chart.")

trend = pd.DataFrame({"Year": year, "Class": pd.to_numeric(df["Class"], errors="coerce")}).dropna()
trend["Year"] = trend["Year"].astype(int)
yearly = trend.groupby("Year", as_index=False)["Class"].mean().rename(columns={"Class":"SuccessRate"})

fig, ax = plt.subplots(figsize=(10, 5.5))
sns.lineplot(data=yearly, x="Year", y="SuccessRate", marker="o", linewidth=2.5, color="#4472C4", ax=ax)
ax.set_title("Falcon 9 Landing Success Rate by Year")
ax.set_xlabel("Year")
ax.set_ylabel("Average Success Rate")
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig("EDA_06_Yearly_Success_Trend.png", bbox_inches="tight")
plt.show()
display(yearly)


## 8. Presentation-ready summary tables

Use these outputs to write the explanations below the six charts in the final PowerPoint.


In [ ]:
site_summary = (df_eda.groupby("LaunchSite")
                .agg(Launches=("Class","size"), Successes=("Class","sum"), SuccessRate=("Class","mean"))
                .sort_values("SuccessRate", ascending=False))
orbit_summary = (df_eda.groupby("Orbit")
                 .agg(Launches=("Class","size"), Successes=("Class","sum"), SuccessRate=("Class","mean"))
                 .sort_values("SuccessRate", ascending=False))

print("Launch site summary")
display(site_summary)
print("Orbit summary")
display(orbit_summary)
print("PNG files created:")
for f in [
    "EDA_01_FlightNumber_vs_LaunchSite.png",
    "EDA_02_Payload_vs_LaunchSite.png",
    "EDA_03_SuccessRate_vs_Orbit.png",
    "EDA_04_FlightNumber_vs_Orbit.png",
    "EDA_05_Payload_vs_Orbit.png",
    "EDA_06_Yearly_Success_Trend.png",
]:
    print(" -", f)


## Next step

After running all cells successfully:

1. Download this executed notebook and replace the old `04_EDA_Visualization.ipynb` in GitHub.
2. Download the six generated PNG files.
3. Insert the PNG files into slides 16–21 of the PowerPoint presentation.
4. Use the summary tables above to write only conclusions supported by your actual data.
